In [2]:
import pickle
from timeit import default_timer as timer
import os
import random

import numpy as np
import umap
from faerun import Faerun
import pandas as pd
import plotly.express as px

def main(fold_len=2048, mit_limit=479035, umap_sample=2000, transform_batch=5000):
    """
    Load DRFP fingerprint pickles for USPTO-MIT and Chan–Lam,
    compute a 2D UMAP embedding (fit on a subset) and transform
    the full dataset in batches, then create an interactive Faerun plot.

    Also write a Plotly HTML scatter so the visualization is usable
    without Faerun.

    Parameters:
    - umap_sample: number of points to fit UMAP on (sampleed from combined data)
    - transform_batch: batch size when calling `umap_model.transform` to limit RAM
    """

    # Build paths for pickled fingerprints (created by earlier cells)
    mit_path = f"fingerprints/drfp_fps_mit_{fold_len}.pkl"
    chanlam_path = f"fingerprints/drfp_fps_chanlam_{fold_len}.pkl"

    if not os.path.exists(mit_path) or not os.path.exists(chanlam_path):
        print("Missing fingerprint files:", mit_path, chanlam_path)
        return

    # Load and sample MIT (shuffle for a fair subset)
    with open(mit_path, "rb") as f:
        fps_mit_full = pickle.load(f)
    fps_mit = fps_mit_full.copy()
    random.shuffle(fps_mit)
    fps_mit = fps_mit[:mit_limit]

    # Load Chan–Lam (use all available)
    with open(chanlam_path, "rb") as f:
        fps_chanlam = pickle.load(f)

    n_mit = len(fps_mit)
    n_chan = len(fps_chanlam)
    n_total = n_mit + n_chan
    print(f"Loaded fingerprints: MIT={n_mit}, Chan-Lam={n_chan}, total={n_total}")

    # Use list concatenation (no large numpy stack yet) so we can process in batches
    combined_list = fps_mit + fps_chanlam

    # Decide sample indices for UMAP fitting
    sample_size = min(umap_sample, n_total)
    sample_idx = random.sample(range(n_total), sample_size) if sample_size < n_total else list(range(n_total))
    sample_fps = np.array([combined_list[i] for i in sample_idx], dtype=float)

    # Fit UMAP on the sample
    start = timer()
    umap_model = umap.UMAP(n_components=2, random_state=42)
    umap_model.fit(sample_fps)
    end = timer()
    print(f"UMAP fit on sample ({sample_size} pts) completed in {end - start:.2f}s.")

    # Transform full dataset in batches to avoid high memory usage
    coords = np.empty((n_total, 2), dtype=float)
    start = timer()
    for i in range(0, n_total, transform_batch):
        j = min(i + transform_batch, n_total)
        batch = np.array([combined_list[k] for k in range(i, j)], dtype=float)
        coords[i:j] = umap_model.transform(batch)
        print(f"Transformed batch {i}-{j} ({j-i} pts)")
    end = timer()
    print(f"UMAP transform of full dataset completed in {end - start:.2f}s.")

    # Build labels and color indices (0 = MIT, 1 = Chan–Lam)
    labels = [f"USPTO-MIT__{i}" for i in range(n_mit)] + [f"Chan-Lam__{i}" for i in range(n_chan)]
    colors_idx = [0] * n_mit + [1] * n_chan
    legend_labels = [(0, "USPTO-MIT"), (1, "Chan-Lam")]

    # Create Faerun plot (interactive HTML)
    faerun = Faerun(view="front", coords=False)
    faerun.add_scatter(
        "drfp_umap",
        {"x": coords[:, 0], "y": coords[:, 1], "c": colors_idx, "labels": labels},
        colormap="tab10",
        categorical=True,
        point_scale=2.0,
        has_legend=True,
        shader="sphere",
        legend_labels=legend_labels,
        title_index=1,
    )

    output_file = f"fingerprint_viz_faerun_{fold_len}"
    print("Saving Faerun visualization to:", output_file)
    faerun.plot(output_file)
    print("Done. Open the generated HTML in a browser to explore.")

    # Also create a Plotly scatter (interactive HTML) so users without Faerun can view the embedding
    try:
        df = pd.DataFrame({
            "x": coords[:, 0],
            "y": coords[:, 1],
            "label": labels,
            "color": colors_idx,
        })
        fig = px.scatter(
            df,
            x="x", y="y", color="color", hover_name="label",
            color_discrete_map={0: "#1f77b4", 1: "#0c7708"},
            title=f"DRFP UMAP (fold {fold_len})"
        )
        output_plotly = f"fingerprint_viz_plotly_{fold_len}.html"
        fig.write_html(output_plotly)
        print("Saved Plotly visualization to:", output_plotly)
    except Exception as e:
        print("Plotly visualization failed:", e)

if __name__ == "__main__":
    main()

Loaded fingerprints: MIT=479035, Chan-Lam=9602, total=488637


c:\Users\Happy\.conda\envs\srp-mt-training\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP fit on sample (2000 pts) completed in 6.10s.
Transformed batch 0-5000 (5000 pts)
Transformed batch 5000-10000 (5000 pts)
Transformed batch 10000-15000 (5000 pts)
Transformed batch 15000-20000 (5000 pts)
Transformed batch 20000-25000 (5000 pts)
Transformed batch 25000-30000 (5000 pts)
Transformed batch 30000-35000 (5000 pts)
Transformed batch 35000-40000 (5000 pts)
Transformed batch 40000-45000 (5000 pts)
Transformed batch 45000-50000 (5000 pts)
Transformed batch 50000-55000 (5000 pts)
Transformed batch 55000-60000 (5000 pts)
Transformed batch 60000-65000 (5000 pts)
Transformed batch 65000-70000 (5000 pts)
Transformed batch 70000-75000 (5000 pts)
Transformed batch 75000-80000 (5000 pts)
Transformed batch 80000-85000 (5000 pts)
Transformed batch 85000-90000 (5000 pts)
Transformed batch 90000-95000 (5000 pts)
Transformed batch 95000-100000 (5000 pts)
Transformed batch 100000-105000 (5000 pts)
Transformed batch 105000-110000 (5000 pts)
Transformed batch 110000-115000 (5000 pts)
Transf

c:\Users\Happy\SRP\SRP Project MT training\data_visualisation\fingerprint_viz_faerun_2048.html

Done. Open the generated HTML in a browser to explore.
Saved Plotly visualization to: fingerprint_viz_plotly_2048.html


In [ ]:
# # Generate fingerprints at multiple folding lengths
# encoder = DrfpEncoder()
# fps_dict = {}  # Will store {folding_length: {dataset: fps}}

# for fold_len in folding_lengths:
#     print(f"\n=== Encoding with folding length {fold_len} ===")
#     fps_mit = encoder.encode(reaction_strings_list, show_progress_bar=True, n_folded_length=fold_len)
#     fps_chanlam = encoder.encode(reaction_strings_list_chanlam, show_progress_bar=True, n_folded_length=fold_len)
#     fps_dict[fold_len] = {"mit": fps_mit, "chanlam": fps_chanlam}
#     print(f"✓ Encoded USPTO-MIT: {len(fps_mit)} fingerprints")
#     print(f"✓ Encoded Chan–Lam: {len(fps_chanlam)} fingerprints")

# # Use 2048 as default for visualization
# fps = fps_dict[2048]["mit"]
# fps2 = fps_dict[2048]["chanlam"]
# print(f"\nDefault fingerprints (2048) sample: {fps[0][:20]}...")
